# 🌍 Veri Toplama Pipeline (Elazığ + İstanbul)

Bu notebook **CAIN-GAN eğitimi için gerçek veri** üretir.

**Yöntem:** Geofabrik bulk OSM extract'i (Overpass API'den çok daha güvenilir)

**Süre:** ~15-30 dakika (ilk kez), sonraki çalıştırmalarda dakikalar

**Hedef çıktı:** `data/{elazig,istanbul}/{train,val,test}/{8 kanal}` — eğitime hazır

---

## Akış

1. Ortam + repo klonlama
2. **Geofabrik Türkiye OSM** (binalar, yollar, su, vejetasyon, landuse) — ~600 MB
3. AFAD sismik veri (proxy / resmi)
4. Copernicus DEM
5. Patch üretimi (256×256)
6. Drive'a kaydet

**Neden Overpass değil?**
- Overpass API büyük şehirlerde 406/429 atıyor (İstanbul'da 1.5M+ bina)
- Geofabrik günlük güncellenen tek bir ZIP — `pip install geopandas` yeterli
- Aynı OSM verisi, %100 güvenilir

## 1️⃣ Kurulum

In [ ]:
# Google Drive bağla (veriyi saklamak için önerilen)
from google.colab import drive
drive.mount('/content/drive')

# Repo'yu klonla
%cd /content
!rm -rf cain-gan-urban-design
!git clone https://github.com/ilker-23/cain-gan-urban-design.git
%cd cain-gan-urban-design

# Bağımlılıklar (geopandas Geofabrik için kritik!)
!pip install -q -r requirements.txt
!pip install -q -r data_collection/requirements.txt
!pip install -q geopandas fiona shapely

print('\n✅ Kurulum tamam')

## 2️⃣ Şehir Seçimi

İlk seferde **Elazığ** ile test edin (daha küçük, daha hızlı).

In [ ]:
# DEĞİŞTİREBİLİRSİNİZ: 'elazig', 'istanbul' veya 'both'
CITY = 'elazig'

# Tüm pipeline mi yoksa tek tek adım mı?
RUN_FULL_PIPELINE = False  # True = tek komutla hepsi, False = adım adım kontrol

print(f'🎯 Hedef şehir: {CITY}')
print(f'   Mod: {"Tam pipeline" if RUN_FULL_PIPELINE else "Adım adım"}')

## 3️⃣ Geofabrik Türkiye OSM (Bulk İndirme)

Bu adım **Overpass yerine** Geofabrik'in günlük güncellenen Türkiye extract'ini indirir.

**Avantajlar:**
- Rate-limit yok, 406 hatası yok
- ~600 MB tek ZIP
- Shapefile formatı (geopandas ile hızlı okuma)
- Binalar, yollar, su, vejetasyon, landuse — hepsi içinde

İlk seferde ~5-10 dk sürer. Sonraki çalıştırmalarda dosya hazır.

In [ ]:
from data_collection.geofabrik_downloader import pipeline as geofabrik_pipeline
from pathlib import Path

raw_root = Path('data_collection/raw')
raw_root.mkdir(parents=True, exist_ok=True)

cities = ['elazig', 'istanbul'] if CITY == 'both' else [CITY]

# Geofabrik bulk pipeline:
#   1. turkey-latest-free.shp.zip indir (~600 MB)
#   2. ZIP çıkar
#   3. Her şehir için BBox ile filtrele
#   4. GeoJSON olarak kaydet (buildings, roads, water, vegetation, landuse)

results = geofabrik_pipeline(cities, output_root=raw_root)

# Sonuç özeti
print('\n\n📊 ÖZET:')
for city, layers in results.items():
    print(f'\n{city.upper()}:')
    for layer, path in layers.items():
        import json
        with open(path) as f:
            n = len(json.load(f).get('features', []))
        print(f'  {layer:12s}: {n:7d} feature')

## 4️⃣ Microsoft Building Footprints (Opsiyonel — Atlanabilir)

Geofabrik zaten binaları içerdiği için bu adım **opsiyonel**.

Eğer daha kapsamlı/yüksek kaliteli bina verisi isterseniz çalıştırın,
yoksa **sonraki hücreye geçin**.

In [ ]:
# ATLA bu hücreyi — Geofabrik zaten binaları içeriyor
# Sadece Microsoft Building Footprints'i de istiyorsanız çalıştırın:

# from data_collection.microsoft_buildings import download_city as ms_download
# for c in cities:
#     ms_download(c, raw_root)
print('⏩ Microsoft adımı atlandı (Geofabrik yeterli)')

## 5️⃣ AFAD Sismik Verisi

In [ ]:
from data_collection.afad_seismic import download_seismic

for c in cities:
    download_seismic(c, raw_root)

print('\n⚠️  AFAD resmi PGA raster\'i için:')
print('   https://tdth.afad.gov.tr/ → akademik talep')

## 6️⃣ Copernicus DEM (30m global)

In [ ]:
from data_collection.copernicus_dem import download_city as dem_download
from data_collection.copernicus_dem import merge_tiles_to_png
from data_collection.config import CITY_BBOXES

for c in cities:
    tiles = dem_download(c, raw_root)
    if tiles:
        out_dir = raw_root / c / 'dem'
        merge_tiles_to_png(tiles, CITY_BBOXES[c]['bbox'],
                            out_dir / 'dem_merged.png', target_size=2048)

## 7️⃣ İBB Açık Veri (sadece İstanbul)

In [ ]:
if 'istanbul' in cities:
    from data_collection.ibb_downloader import download_layer
    for layer in ['zoning', 'microzoning']:
        try:
            download_layer(layer, max_datasets=2)
        except Exception as e:
            print(f'⚠️  {layer}: {e}')
else:
    print('⏩ İstanbul yok, atlanıyor')

## 8️⃣ Raw Veri Kontrolü

In [ ]:
import json
from pathlib import Path

for c in cities:
    print(f'\n📊 {c.upper()}:')
    city_dir = Path('data_collection/raw') / c
    
    for layer in ['buildings', 'roads', 'water', 'vegetation', 'landuse']:
        path = city_dir / f'{layer}.geojson'
        if path.exists():
            with open(path) as f:
                data = json.load(f)
            n_feat = len(data.get('features', []))
            size_mb = path.stat().st_size / 1e6
            print(f'  {layer:15s}: {n_feat:6d} feature, {size_mb:.2f} MB')
        else:
            print(f'  {layer:15s}: ❌ yok')
    
    # Microsoft
    ms_path = city_dir / 'microsoft_buildings' / 'buildings_microsoft.geojson'
    if ms_path.exists():
        with open(ms_path) as f:
            data = json.load(f)
        print(f'  microsoft     : {len(data["features"]):6d} bina')
    
    # Sismik + DEM
    for sub in ['seismic/pga_475yr.png', 'dem/dem_merged.png']:
        path = city_dir / sub
        if path.exists():
            print(f'  {sub:15s}: ✅')
        else:
            print(f'  {sub:15s}: ❌')

## 🔍 ÖN TEŞHİS — Patch Üretmeden Önce

Bu adım kritik:
- BBox'unuz yoğun urban alana mı düşüyor?
- Kaç patch geçerli olacak?
- Bina yoğunluğu nasıl dağılmış?

⚠️ Eğer %20'den az patch geçerliyse `config.py`'da BBox veya min_buildings ayarı gerekir.

In [ ]:
from data_collection.diagnose_data import diagnose_pre_patching
from pathlib import Path

for c in cities:
    stats = diagnose_pre_patching(c, Path('data_collection/raw'),
                                    Path('diagnostics'))
    valid_ratio = stats.get('valid_ratio', 0)
    if valid_ratio < 0.05:
        print(f'\n⚠️  {c}: çok az geçerli patch ({valid_ratio*100:.1f}%)')
        print('   config.py\'da BBox\'u daraltın veya min_buildings\'i düşürün')

## 9️⃣ Patch Üretimi

Bu adım **CAIN-GAN training input'unu** üretir.

In [ ]:
from data_collection.patch_extractor import process_city
from pathlib import Path

data_output = Path('data')
data_output.mkdir(exist_ok=True)

for c in cities:
    stats = process_city(c, Path('data_collection/raw'), data_output)
    print(f'\n✅ {c}: {stats}')

## 🔟 Üretilen Patch'leri Görselleştir

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import random

fig, axes = plt.subplots(len(cities), 8, figsize=(24, 4 * len(cities)))
if len(cities) == 1:
    axes = axes.reshape(1, -1)

channels = ['site_context', 'planning_guidance', 'neighboring_footprints',
            'mask', 'seismic', 'dem', 'footprint_target', 'height_target']

for row, c in enumerate(cities):
    train_dir = Path(f'data/{c}/train/site_context')
    if not train_dir.exists():
        print(f'⚠️ {c}/train yok')
        continue
    samples = list(train_dir.glob('*.png'))
    if not samples:
        continue
    sample_id = random.choice(samples).stem
    
    for col, ch in enumerate(channels):
        path = Path(f'data/{c}/train/{ch}/{sample_id}.png')
        if path.exists():
            img = Image.open(path)
            cmap = 'gray'
            if ch == 'planning_guidance':
                img = img.convert('RGB')
                cmap = None
            elif ch == 'seismic':
                cmap = 'hot'
            elif ch == 'dem':
                cmap = 'terrain'
            
            axes[row, col].imshow(img, cmap=cmap)
            axes[row, col].set_title(f'{c} — {ch}', fontsize=10)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('real_data_visualization.png', dpi=100, bbox_inches='tight')
plt.show()
print('💾 Kaydedildi: real_data_visualization.png')

In [ ]:
# 8 rastgele patch'in tüm 8 kanalını gösteren detaylı görselleştirme
from data_collection.diagnose_data import diagnose_post_patching
from pathlib import Path

for c in cities:
    diagnose_post_patching(c, Path('data'), Path('diagnostics'), n_samples=8)

# Drive'a kopyala
import shutil
for diag_file in Path('diagnostics').glob('*.png'):
    shutil.copy(diag_file, '/content/drive/MyDrive/' + diag_file.name)
    print(f'💾 Drive: {diag_file.name}')

## 1️⃣1️⃣ Drive'a Kaydet (Önemli — Colab session kapanmadan)

In [ ]:
import shutil
from pathlib import Path

DRIVE_OUTPUT = Path('/content/drive/MyDrive/cain_gan_data')
DRIVE_OUTPUT.mkdir(exist_ok=True)

# Patch'leri Drive'a kopyala (büyük ama önemli)
if Path('data').exists():
    print('📤 Patch\'ler Drive\'a kopyalanıyor...')
    shutil.copytree('data', DRIVE_OUTPUT / 'data', dirs_exist_ok=True)
    print(f'✅ {DRIVE_OUTPUT}/data')

# Raw veri (opsiyonel, çok büyük olabilir)
SAVE_RAW = False  # True yaparsanız raw veri de kaydedilir
if SAVE_RAW and Path('data_collection/raw').exists():
    print('📤 Raw veri Drive\'a kopyalanıyor (büyük!)...')
    shutil.copytree('data_collection/raw', DRIVE_OUTPUT / 'raw', dirs_exist_ok=True)
    print(f'✅ {DRIVE_OUTPUT}/raw')

# Görseller
for img in Path('.').glob('*.png'):
    shutil.copy(img, DRIVE_OUTPUT / img.name)

print('\n🎉 Drive backup tamam!')

## 🎓 Sonraki Adım: Gerçek Veri ile Eğitim

Şimdi `CAIN_GAN_Colab.ipynb` notebook'unu açın ve:

1. **Synthetic veri oluşturma hücresini atlayın** (zaten gerçek verimiz var)
2. **Drive'dan veriyi yükleyin:**
   ```python
   !cp -r /content/drive/MyDrive/cain_gan_data/data ./data
   ```
3. **Epoch sayısını artırın:**
   ```python
   trainer.train(num_epochs_footprint=50, num_epochs_height=50)
   ```

---

## ⚠️ Önemli Notlar

1. **Resmi AFAD verisi** kullanmak SCI dergi için gerekli (proxy yeterli değil)
2. **Elazığ Belediyesi** imar planları için akademik başvuru yapın
3. **Etik kurul onayı** alın (özellikle bina verisinde)
4. **Veri atıfları** zorunlu — `data_collection/README.md` bölümüne bakın

---

## 📊 Beklenen Veri Hacmi

| Şehir | Patch Sayısı | Drive Boyutu |
|-------|--------------|--------------|
| Elazığ | 1.500-3.000 | ~150 MB |
| İstanbul | 8.000-15.000 | ~750 MB |

🎓 Başarılar!